# z024 - Pruebas de la noche (10/10): una modificación por vez contra la referencia

**Referencia:** trial 34 de z005, target juntar (BAJA+1 + BAJA+2), 761 variables (originales + lags/deltas 1 y 2).
10 semillas × 2 cortes: train 202104 → valid 202106 ("junio") y train 202103 → valid 202105 ("mayo").

| Versión | Cambio |
|---|---|
| a_sin_prestamos | sin mprestamos_personales y cprestamos_personales (y sus lags/deltas) |
| b_sin_fechas | sin Master_fultimo_cierre y Visa_fultimo_cierre (y sus lags/deltas) |
| ab_sin_prest_y_fechas | a + b juntas |
| c_solo_baja2 | target solo BAJA+2 |
| d_paso_a_cero | + flags "pasó a cero" (mes anterior > 0 y este mes 0): mpayroll, cpayroll_trx, mcaja_ahorro, ctrx_quarter |
| e_rankings | + ranking con cero fijo dentro de cada mes (negativos −1 a 0, cero 0, positivos 0 a 1) y su delta contra el mes anterior, para las variables en pesos del top 30 de z022 |
| f_tendencias | + media de 3 meses y cociente valor / media (como z014) |

**Métricas** (ganancia evaluada siempre con BAJA+2): ganancia máxima y ganancia en **N fijo = 7,29% de los clientes del mes**
(la proporción de 12.000 / 164.647 del submit), semillas ganadas contra la referencia (pareadas por semilla) y p-valor de Wilcoxon.

**Regla de adopción (definida antes de correr):**
- c, d, e, f: gana en **≥ 8/10 semillas en los dos cortes** (ganancia máxima) **y** la ganancia media en N fijo mejora en los dos.
- a, b, ab: igual pero **solo mayo** (junio queda informativo por la ola de salidas de clientes con préstamo en agosto)
  + chequeo de exposición (submit 28: 6,8% de los 12.000 está en top 10% de préstamo o tiene 3+, contra 12,1% de la cartera → exposición chica).

**Ojo en e:** el top 30 sale de z022, un modelo entrenado con 202103-202106 (incluye los meses de validación) → fuga chica en la elección de variables, aceptada.

Cada predicción se guarda en `probs_z024/`: si se corta, volver a correr y retoma. Progreso en `z024_progreso.txt`.

In [1]:
# --- 1. Librerías y parámetros ---
import os, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import wilcoxon

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
SEMILLAS = [270131, 417293, 583117, 692347, 851071,    # las del submit 16-17
            135791, 246809, 314159, 527183, 964201]    # 5 nuevas
CORTES = {"junio": (202104, 202106), "mayo": (202103, 202105)}   # nombre: (mes train, mes valid)
PROP_N = 12_000 / 164_647          # proporción del submit: 12.000 de los 164.647 clientes de 202108
CARPETA = "probs_z024"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}
elegidos = pd.read_csv("z005_top10_final.csv", sep=";", decimal=",", encoding="utf-8-sig", index_col="trial")
trial = elegidos.loc[34]
params = {**params_fijos, "num_leaves": int(trial["params_num_leaves"]),
          "min_sum_hessian_in_leaf": float(trial["params_min_sum_hessian_in_leaf"])}
N_ARBOLES = int(trial["user_attrs_best_iter"])

def log(texto):
    # El notebook no muestra nada hasta terminar: dejo el avance en un archivo de texto
    with open("z024_progreso.txt", "a", encoding="utf-8") as f:
        f.write(time.strftime("%H:%M:%S ") + texto + "\n")
log(f"inicio | árboles {N_ARBOLES}")
params, N_ARBOLES

({'objective': 'binary',
  'learning_rate': 0.005,
  'min_data_in_leaf': 0,
  'feature_fraction': 0.5,
  'feature_fraction_bynode': 0.2,
  'max_bin': 31,
  'verbose': -1,
  'num_leaves': 137,
  'min_sum_hessian_in_leaf': 9.755574121446362},
 538)

In [2]:
# --- 2. Datos (solo 202103-202106: lags y medias miran hacia atrás, 07 y 08 no hacen falta) + lags/deltas ---
df = pd.read_parquet("../competencia_01.parquet")
df = df[df["foto_mes"] <= 202106]
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
df["y_juntar"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
df["y_solo"]   = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "y_juntar", "y_solo"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]   # las originales

grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

base = [c for c in df.columns if c not in no_usar]    # las 761 de siempre
print("Variables base:", len(base))

Variables base: 761


In [3]:
# --- 3. Variables nuevas de d, e y f ---
# d. Flags "pasó a cero": tenía valor > 0 el mes anterior y este mes 0
flags = pd.DataFrame({f"{c}_paso_a_cero": ((df[f"{c}_lag1"] > 0) & (df[c] == 0)).astype("int8")
                      for c in ["mpayroll", "cpayroll_trx", "mcaja_ahorro", "ctrx_quarter"]})

# e. Ranking con cero fijo, dentro de cada mes, para las variables en pesos del top 30 de z022
imp = pd.read_csv("z022_importancia.csv", sep=";", decimal=",", encoding="utf-8-sig")
bases_top = imp.head(30)["variable"].str.replace(r"_(lag|delta)\d$", "", regex=True)   # ctrx_quarter_lag1 -> ctrx_quarter
en_pesos = [v for v in dict.fromkeys(bases_top) if v.startswith(("m", "Visa_m", "Master_m"))]
print(len(en_pesos), "variables en pesos:", en_pesos)

def rank_cero_fijo(s):
    # Ejemplo: [-500, 0, 10, 20] -> [-1, 0, 0.5, 1]. El cero queda siempre en 0, aunque cambie la inflación.
    out = pd.Series(np.where(s == 0, 0.0, np.nan), index=s.index)
    pos, neg = s > 0, s < 0
    out[pos] = s[pos].rank(pct=True)          # positivos: (0, 1]
    out[neg] = -(-s[neg]).rank(pct=True)      # negativos: el más negativo = -1
    return out

rk = {}
for v in en_pesos:
    r_mes = df.groupby("foto_mes")[v].transform(rank_cero_fijo).astype("float32")
    rk[f"{v}_rank"] = r_mes
    rk[f"{v}_rank_delta1"] = (r_mes - r_mes.groupby(df["numero_de_cliente"]).shift(1)).astype("float32")
rankings = pd.DataFrame(rk)
del rk

# f. Tendencias (igual que z014): media de los últimos 3 meses y cociente valor / media
media3 = (df.groupby("numero_de_cliente")[cols].rolling(3, min_periods=1).mean()
            .reset_index(level=0, drop=True).sort_index().astype("float32"))
ratio3 = (df[cols] / media3.replace(0, np.nan)).astype("float32")
media3.columns = [f"{c}_media3" for c in cols]
ratio3.columns = [f"{c}_ratio3" for c in cols]

df = pd.concat([df, flags, rankings, media3, ratio3], axis=1)
nuevas_d, nuevas_e = list(flags.columns), list(rankings.columns)
nuevas_f = list(media3.columns) + list(ratio3.columns)
del flags, rankings, media3, ratio3
log("variables listas")

16 variables en pesos: ['mpasivos_margen', 'mcuentas_saldo', 'mcaja_ahorro', 'mpayroll', 'mtarjeta_visa_consumo', 'mautoservicio', 'mprestamos_personales', 'Visa_msaldopesos', 'Visa_mpagominimo', 'mcomisiones_mantenimiento', 'mcuenta_corriente', 'Visa_mpagospesos', 'mrentabilidad_annual', 'Visa_msaldototal', 'mtransferencias_recibidas', 'mactivos_margen']


In [4]:
# --- 4. Versiones (una sola modificación cada una) ---
def sin(prefijos):
    return [c for c in base if not c.startswith(prefijos)]

PREST  = ("mprestamos_personales", "cprestamos_personales")
FECHAS = ("Master_fultimo_cierre", "Visa_fultimo_cierre")

VERSIONES = {                              # nombre: (variables, target de entrenamiento)
    "ref":                   (base, "y_juntar"),
    "a_sin_prestamos":       (sin(PREST), "y_juntar"),
    "b_sin_fechas":          (sin(FECHAS), "y_juntar"),
    "ab_sin_prest_y_fechas": (sin(PREST + FECHAS), "y_juntar"),
    "c_solo_baja2":          (base, "y_solo"),
    "d_paso_a_cero":         (base + nuevas_d, "y_juntar"),
    "e_rankings":            (base + nuevas_e, "y_juntar"),
    "f_tendencias":          (base + nuevas_f, "y_juntar"),
}
SOLO_MAYO = ["a_sin_prestamos", "b_sin_fechas", "ab_sin_prest_y_fechas"]   # regla de adopción con mayo
for v, (vs, t) in VERSIONES.items():
    print(f"{v:24s} {len(vs):5d} variables, target {t}")

ref                        761 variables, target y_juntar
a_sin_prestamos            751 variables, target y_juntar
b_sin_fechas               751 variables, target y_juntar
ab_sin_prest_y_fechas      741 variables, target y_juntar
c_solo_baja2               761 variables, target y_solo
d_paso_a_cero              765 variables, target y_juntar
e_rankings                 793 variables, target y_juntar
f_tendencias              1065 variables, target y_juntar


In [5]:
# --- 5. Entrenar (8 versiones × 2 cortes × 10 semillas = 160 modelos; retoma lo que ya está guardado) ---
foto = df["foto_mes"].values
for corte, (mes_train, mes_valid) in CORTES.items():
    tr, va = foto == mes_train, foto == mes_valid
    for version, (vs, target) in VERSIONES.items():
        faltan = [s for s in SEMILLAS if not os.path.exists(f"{CARPETA}/{version}_{corte}_{s}.npy")]
        if not faltan:
            continue
        datos = lgb.Dataset(df.loc[tr, vs], label=df.loc[tr, target])
        X_valid = df.loc[va, vs]
        for s in faltan:
            inicio = time.time()
            modelo = lgb.train({**params, "seed": s}, datos, num_boost_round=N_ARBOLES)
            np.save(f"{CARPETA}/{version}_{corte}_{s}.npy", modelo.predict(X_valid).astype("float32"))
            log(f"{corte} {version} semilla {s}: {time.time() - inicio:.0f} s")
        del datos, X_valid
log("entrenamiento terminado")

In [6]:
# --- 6. Ganancias por semilla ---
filas = []
for corte, (mes_train, mes_valid) in CORTES.items():
    va = foto == mes_valid
    gan = np.where(df.loc[va, "clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    n_fijo = round(PROP_N * va.sum())
    for version in VERSIONES:
        for s in SEMILLAS:
            prob = np.load(f"{CARPETA}/{version}_{corte}_{s}.npy")
            acum = np.cumsum(gan[np.argsort(-prob)])        # ganancia si mando los N con mayor probabilidad
            filas.append({"corte": corte, "version": version, "semilla": s, "n_fijo": n_fijo,
                          "gan_max": acum.max() / 1e6, "mejor_n": acum.argmax() + 1,
                          "gan_nfijo": acum[n_fijo - 1] / 1e6, "gan_12000": acum[11_999] / 1e6})
por_semilla = pd.DataFrame(filas)
por_semilla.to_csv("z024_por_semilla.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
por_semilla.groupby(["corte", "n_fijo"]).size()

corte  n_fijo
junio  11961     80
mayo   11936     80
dtype: int64

In [7]:
# --- 7. Comparación contra la referencia (pareada por semilla) y regla de adopción ---
ref = por_semilla[por_semilla["version"] == "ref"].set_index(["corte", "semilla"])
res = []
for (corte, version), g in por_semilla.groupby(["corte", "version"], sort=False):
    g = g.set_index(["corte", "semilla"])
    d_max, d_fijo = g.loc[corte, "gan_max"] - ref.loc[corte, "gan_max"], g.loc[corte, "gan_nfijo"] - ref.loc[corte, "gan_nfijo"]   # solo este corte (antes mezclaba los dos y el Wilcoxon daba NaN)
    es_ref = version == "ref"
    res.append({"corte": corte, "version": version,
                "gan_max": g["gan_max"].mean(), "gan_max_desv": g["gan_max"].std(),
                "gan_nfijo": g["gan_nfijo"].mean(), "gan_nfijo_desv": g["gan_nfijo"].std(),
                "mejor_n": g["mejor_n"].mean(),
                "dif_max": d_max.mean(), "dif_nfijo": d_fijo.mean(),
                "gana_max": np.nan if es_ref else (d_max > 0).sum(),
                "gana_nfijo": np.nan if es_ref else (d_fijo > 0).sum(),
                "p_wilcoxon_max": np.nan if es_ref else (np.nan if (d_max == 0).all() else wilcoxon(d_max).pvalue),
                "p_wilcoxon_nfijo": np.nan if es_ref else (np.nan if (d_fijo == 0).all() else wilcoxon(d_fijo).pvalue)})
res = pd.DataFrame(res).set_index(["version", "corte"])

def adopta(version):
    if version == "ref":
        return ""
    cortes = ["mayo"] if version in SOLO_MAYO else ["junio", "mayo"]
    ok = all(res.loc[(version, c), "gana_max"] >= 8 and res.loc[(version, c), "dif_nfijo"] > 0 for c in cortes)
    return ("SÍ" if ok else "no") + (" (decide mayo)" if version in SOLO_MAYO else " (ambos cortes)")

tabla = res.round(3).unstack("corte")
tabla.columns = [f"{m}_{c}" for m, c in tabla.columns]
orden = [f"{m}_{c}" for c in ["mayo", "junio"] for m in
         ["gan_max", "gan_max_desv", "dif_max", "gana_max", "p_wilcoxon_max", "gan_nfijo", "dif_nfijo", "gana_nfijo", "p_wilcoxon_nfijo", "mejor_n"]]
tabla = tabla.loc[list(VERSIONES), orden]
tabla["adopta"] = [adopta(v) for v in tabla.index]
tabla.to_csv("z024_resultados.csv", sep=";", decimal=",", encoding="utf-8-sig", index_label="version")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
tabla

,gan_max_mayo,gan_max_desv_mayo,dif_max_mayo,gana_max_mayo,p_wilcoxon_max_mayo,gan_nfijo_mayo,dif_nfijo_mayo,gana_nfijo_mayo,p_wilcoxon_nfijo_mayo,mejor_n_mayo,gan_max_junio,gan_max_desv_junio,dif_max_junio,gana_max_junio,p_wilcoxon_max_junio,gan_nfijo_junio,dif_nfijo_junio,gana_nfijo_junio,p_wilcoxon_nfijo_junio,mejor_n_junio,adopta
version,,,,,,,,,,,,,,,,,,,,,
ref,274.466,1.980,0.000,NaN,NaN,250.58,0.00,NaN,NaN,8011.4,418.828,3.443,0.000,NaN,NaN,387.502,0.00,NaN,NaN,14769.9,
a_sin_prestamos,250.940,1.739,-23.526,0.0,NaN,232.54,-18.04,0.0,NaN,8426.9,457.163,2.500,38.335,10.0,NaN,451.742,64.24,10.0,NaN,12691.9,no (decide mayo)
b_sin_fechas,274.898,1.930,0.432,6.0,NaN,247.83,-2.75,2.0,NaN,7335.7,418.473,4.201,-0.355,5.0,NaN,388.162,0.66,4.0,NaN,14366.8,no (decide mayo)
ab_sin_prest_y_fechas,249.579,1.447,-24.887,0.0,NaN,231.11,-19.47,0.0,NaN,8468.4,459.453,3.002,40.626,10.0,NaN,454.712,67.21,10.0,NaN,12088.6,no (decide mayo)
c_solo_baja2,253.619,2.417,-20.848,0.0,NaN,231.33,-19.25,0.0,NaN,7601.5,400.097,3.542,-18.730,0.0,NaN,391.572,4.07,7.0,NaN,13595.0,no (ambos cortes)
d_paso_a_cero,274.466,1.980,0.000,0.0,NaN,250.58,0.00,0.0,NaN,8011.4,417.368,3.079,-1.460,3.0,NaN,388.052,0.55,5.0,NaN,14835.0,no (ambos cortes)
e_rankings,273.171,3.318,-1.295,4.0,NaN,251.68,1.10,6.0,NaN,8546.5,414.112,4.234,-4.716,2.0,NaN,384.643,-2.86,3.0,NaN,14509.4,no (ambos cortes)
f_tendencias,261.750,2.110,-12.716,0.0,NaN,236.83,-13.75,0.0,NaN,7545.8,418.646,4.246,-0.181,3.0,NaN,392.232,4.73,8.0,NaN,15092.5,no (ambos cortes)


In [8]:
# --- 8. Anotar en resultados_experimentos.csv (una fila por versión y corte) ---
ruta = "resultados_experimentos.csv"
hist = pd.read_csv(ruta, sep=";", decimal=",", encoding="utf-8-sig")
hist = hist[~hist["descripcion"].str.startswith("z024")]      # sin duplicar si se vuelve a correr
nuevas_filas = []
for (version, corte), r in res.iterrows():
    mes_train, mes_valid = CORTES[corte]
    g12 = por_semilla.query("version == @version and corte == @corte")["gan_12000"].mean()
    extra = "" if version == "ref" else (f"; dif máx {r['dif_max']:+.1f} M, gana {int(r['gana_max'])}/10, p={r['p_wilcoxon_max']:.3f}; "
                                         f"dif N fijo {r['dif_nfijo']:+.1f} M, gana {int(r['gana_nfijo'])}/10; adopta: {adopta(version)}")
    nuevas_filas.append({
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z024 {version} ({corte}): trial 34, 10 semillas; N fijo {r['gan_nfijo']:.1f} M" + extra,
        "meses_train": str([mes_train]), "mes_valid": mes_valid, "semilla": str(SEMILLAS), "params": str(params),
        "mejor_n": int(r["mejor_n"]), "ganancia_max_M": round(r["gan_max"], 1), "ganancia_top12000_M": round(g12, 1)})
hist = pd.concat([hist, pd.DataFrame(nuevas_filas)], ignore_index=True)
hist["n_enviados"] = hist["n_enviados"].astype("Int64")
hist.to_csv(ruta, sep=";", decimal=",", index=False, encoding="utf-8-sig")
log("todo listo")
print("Anotado en", ruta)

Anotado en resultados_experimentos.csv


## Conclusiones (11/10, 02:20) — ninguna prueba pasa la regla de adopción

Diferencia contra la referencia (juntar, 761 variables), 10 semillas. Ganancia máx / N fijo (M), semillas ganadas, p Wilcoxon (máx):

| Versión | Mayo | Junio | Adopta |
|---|---|---|---|
| a sin préstamos | −23,5 / −18,0, 0/10, p=0,002 | +38,3 / +64,2, 10/10 (ola) | **no** (decide mayo) |
| b sin fechas | +0,4 / −2,8, 6/10, p=0,63 | −0,4 / +0,7, 5/10 | **no** |
| a+b | −24,9 / −19,5, 0/10, p=0,002 | +40,6 / +67,2, 10/10 (ola) | **no** |
| c solo BAJA+2 | −20,8 / −19,3, 0/10, p=0,002 | −18,7 / +4,1, 0/10, p=0,002 | **no** → juntar confirmado |
| d pasó a cero | idéntico a ref (ver nota) | −1,5 / +0,6, 3/10 | **no** |
| e rankings | −1,3 / +1,1, 4/10, p=0,49 | −4,7 / −2,9, 2/10, p=0,02 | **no** |
| f tendencias | −12,7 / −13,8, 0/10 (ver nota) | −0,2 / +4,7, 3/10 | **no** |

- **Préstamos**: en un mes normal (mayo) son de las variables más útiles (−23 M sin ellas). En junio estorban por la ola de
  salidas de clientes con préstamo en agosto. Se quedan. Exposición del submit 28: 6,8% vs 12,1% de la cartera (chica).
- **Juntar vs solo BAJA+2**: juntar gana 10/10 en mayo y en junio (ganancia máx), p = 0,002. Tercera confirmación (z009, z019).
- **Nota d y f en mayo**: el corte de mayo entrena con 202103, el primer mes de los datos → no tiene lags (todos NaN).
  El flag "pasó a cero" queda constante en 0 en el train (modelo idéntico a ref) y la media de 3 meses es igual al valor
  (cociente = 1) en el train pero no en la validación → el −12,7 M de f en mayo es en parte un artefacto. En junio f empata.
- **Fuga chica en e** (aceptada): el top 30 de z022 salió de un modelo entrenado con 202103-202106, incluye los meses de validación.
- Bug corregido después de correr: los p-valores salían NaN (se mezclaban los dos cortes al restar). Recalculados sin reentrenar.
